# EDA Steam · F2P vs Pago
**¿En qué se diferencian los juegos Free-to-Play (F2P) de los juegos de pago en Steam?**

Analisis sobre el catalogo completo de Steam (122,611 juegos brutos, 58,251 tras filtrado por traccion).

---

## Hipotesis

1. **H1 - Recepcion:** Entre juegos con >=50 resenas, los F2P obtienen un % medio de positivas menor que los de pago.
2. **H2 - Generos:** La distribucion de generos entre F2P y de pago es significativamente distinta.
3. **H3 - Volumen:** Los F2P tienen una mediana de Peak CCU mayor que los de pago.
4. **H4 - Tendencia:** El % de lanzamientos F2P sobre el total ha crecido cada anyo desde 2010.

## Resumen ejecutivo de resultados

| Hip. | Veredicto | Evidencia |
|------|-----------|-----------|
| H1 | **CONFIRMADA** | F2P med 79.17% vs Pago 82.4% · Mann-Whitney p=1.89e-32 |
| H2 | **CONFIRMADA** | Chi2 = 4135 · p=0.00e+00 |
| H3 | **REFUTADA** | Medianas = 0 en ambos grupos; sin embargo 5 de los 10 juegos con mas CCU son F2P |
| H4 | **REFUTADA** | Curva en U invertida: 8,6% (2010) -> 22,4% (2020) -> 7,3% (2024) · rho=-0.02 |


## 0 · Setup

In [ ]:
import sys, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titleweight"] = "bold"

# Hacer importable src/
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.data_loader import (
    load_raw, clean_steam, add_features, resumen_calidad
)

IMG_DIR = ROOT / "src" / "img"
IMG_DIR.mkdir(parents=True, exist_ok=True)
PALETTE = {"F2P": "#7dd4de", "Pago": "#2a8a9a"}

print("Setup OK")


---
## Fase 1 · Business Case & Data

**Problema:** Steam aloja juegos F2P y juegos de pago. Las creencias del sector sobre cual funciona en cada nicho se apoyan en casos famosos. Queremos contrastarlas con el catalogo completo.

**Dataset:** Steam Games Dataset (FronkonGames), snapshot enero 2026. Disponible en HuggingFace y Kaggle. **Nota:** el CSV tiene un bug en el header (39 columnas vs 40 reales). `data_loader.py` lo corrige automaticamente.

In [ ]:
df_raw = load_raw()
print(f"Raw: {df_raw.shape[0]:,} juegos x {df_raw.shape[1]} columnas")
df_raw.head(3)


---
## Fase 2 · Data Understanding

In [ ]:
df_raw.info()


In [ ]:
# Resumen de calidad (Top 10 columnas por % nulos)
calidad = resumen_calidad(df_raw).sort_values("pct_nulos", ascending=False)
calidad.head(15)


**Observaciones clave:**

- Columnas inutiles por exceso de nulos: `Movies` (100%), `Score rank` (~100%), `Metacritic url` (96%), `Reviews` (90%), `Notes` (82%).
- Variables clave (Price, Positive, Negative, Peak CCU, Estimated owners) sin nulos.
- **Alarma:** 66,8% del catalogo bruto aparece como F2P (Price=0). El consenso del sector es ~25-30%. Hay que filtrar.

---
## Fase 3 · Data Cleaning

- Eliminamos columnas con >50% de nulos o sin valor analitico.
- Filtramos por **traccion minima**: al menos 10 resenas o Peak CCU > 0.
- Anadimos variables derivadas: `modelo_negocio`, `total_reviews`, `pct_positivas`, `anyo_lanzamiento`, `genero_principal`.

In [ ]:
df = clean_steam(df_raw, filtrar_traccion=True, min_reviews=10)
df = add_features(df)

print(f"Tras limpieza: {df.shape[0]:,} juegos x {df.shape[1]} columnas")
print(f"  F2P:  {(df['modelo_negocio']=='F2P').sum():,}  ({(df['modelo_negocio']=='F2P').mean()*100:.1f}%)")
print(f"  Pago: {(df['modelo_negocio']=='Pago').sum():,}  ({(df['modelo_negocio']=='Pago').mean()*100:.1f}%)")


> El filtrado reduce los F2P del 66,8% al **13,0%**, una proporcion mucho mas alineada con la realidad observable del catalogo.

---
## Fase 4 · Analisis Exploratorio

### 4.1 · Univariante

In [ ]:
# Distribucion del modelo de negocio
fig, ax = plt.subplots(figsize=(8,5))
vc = df["modelo_negocio"].value_counts()
ax.bar(vc.index, vc.values, color=[PALETTE[x] for x in vc.index])
ax.set_title("Distribucion de modelo de negocio (tras filtrado)")
ax.set_ylabel("Numero de juegos")
for i, v in enumerate(vc.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.savefig(IMG_DIR / "01_univariante_modelo_negocio.png", dpi=120)
plt.show()


In [ ]:
# Distribucion de precios (solo juegos de pago)
precio_pago = df.loc[df["modelo_negocio"]=="Pago", "Price"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(precio_pago, bins=40, ax=axes[0], color="#2a8a9a")
axes[0].set_title("Distribucion de precios (juegos de pago)")
axes[0].set_xlabel("Precio (USD)")
sns.boxplot(x=precio_pago, ax=axes[1], color="#2a8a9a")
axes[1].set_title("Boxplot de precios (juegos de pago)")
plt.tight_layout()
plt.savefig(IMG_DIR / "02_univariante_precios.png", dpi=120)
plt.show()

print(precio_pago.describe().round(2))


In [ ]:
# Distribucion del % de positivas
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df["pct_positivas"].dropna(), bins=40, color="#4ab8c8", ax=ax)
ax.set_title("Distribucion del % de resenas positivas")
ax.set_xlabel("% positivas")
plt.tight_layout()
plt.savefig(IMG_DIR / "03_univariante_pct_positivas.png", dpi=120)
plt.show()


### 4.2 · Bivariante por hipotesis

#### H1 - Recepcion: %positivas F2P < Pago (>=50 reviews)

**Veredicto: CONFIRMADA** · F2P med 79.17% vs Pago 82.4% · Mann-Whitney p = 1.89e-32

In [ ]:
df_h1 = df[df["total_reviews"] >= 50].copy()
a = df_h1.loc[df_h1["modelo_negocio"]=="F2P",  "pct_positivas"].dropna()
b = df_h1.loc[df_h1["modelo_negocio"]=="Pago", "pct_positivas"].dropna()
u, p = stats.mannwhitneyu(a, b, alternative="less")

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df_h1, x="modelo_negocio", y="pct_positivas", palette=PALETTE, ax=ax)
ax.set_title(f"H1: %positivas por modelo (>=50 reviews)\nF2P n={len(a):,} med={a.median():.1f}% | Pago n={len(b):,} med={b.median():.1f}% | p={p:.2e}")
ax.set_ylabel("% positivas"); ax.set_xlabel("")
plt.tight_layout()
plt.savefig(IMG_DIR / "h1_pct_positivas.png", dpi=120)
plt.show()


#### H2 - Generos: distribucion distinta entre F2P y Pago

**Veredicto: CONFIRMADA** · Chi2 = 4135, dof = 27, p = 0.00e+00

In [ ]:
tabla = pd.crosstab(df["genero_principal"], df["modelo_negocio"])
chi2, p_h2, dof, _ = stats.chi2_contingency(tabla)

top_f2p = df.loc[df["modelo_negocio"]=="F2P",  "genero_principal"].value_counts().head(10)
top_pago= df.loc[df["modelo_negocio"]=="Pago", "genero_principal"].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
top_f2p.plot(kind="barh", ax=axes[0], color="#7dd4de")
axes[0].invert_yaxis(); axes[0].set_title("Top 10 generos - F2P"); axes[0].set_xlabel("Numero de juegos")
top_pago.plot(kind="barh", ax=axes[1], color="#2a8a9a")
axes[1].invert_yaxis(); axes[1].set_title("Top 10 generos - Pago"); axes[1].set_xlabel("Numero de juegos")
plt.suptitle(f"H2 · Chi2 p-value = {p_h2:.2e}")
plt.tight_layout()
plt.savefig(IMG_DIR / "h2_top_generos.png", dpi=120)
plt.show()


#### H3 - Peak CCU: F2P > Pago (mediana)

**Veredicto: REFUTADA** · Mediana = 0 en ambos. Pero el top del catalogo lo dominan F2P:

In [ ]:
# Top 10 por Peak CCU
top_ccu = df.nlargest(10, "Peak CCU")[["Name","modelo_negocio","Peak CCU","Price"]]
print(top_ccu.to_string(index=False))


In [ ]:
mask_ccu = df["Peak CCU"] > 0
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df[mask_ccu], x="modelo_negocio", y="Peak CCU", palette=PALETTE, ax=ax)
ax.set_yscale("log")
ax.set_title("H3: Peak CCU por modelo (log, CCU>0)")
ax.set_ylabel("Peak CCU (log)"); ax.set_xlabel("")
plt.tight_layout()
plt.savefig(IMG_DIR / "h3_peak_ccu.png", dpi=120)
plt.show()


#### H4 - Evolucion temporal: % F2P crece desde 2010

**Veredicto: REFUTADA** · La hipotesis original era monotonia creciente, pero la realidad es una **curva en U invertida**: 8,6% (2010) -> 22,4% (2020, pico pandemia) -> 7,3% (2024). Spearman rho = -0.02.

In [ ]:
df_t = df[df["anyo_lanzamiento"].between(2010, 2024)]
por_anyo = df_t.groupby(["anyo_lanzamiento","modelo_negocio"]).size().unstack(fill_value=0)
por_anyo["total"] = por_anyo.sum(axis=1)
por_anyo["pct_f2p"] = por_anyo["F2P"] / por_anyo["total"] * 100

spear = stats.spearmanr(por_anyo.index, por_anyo["pct_f2p"])

fig, ax = plt.subplots(figsize=(11, 5))
por_anyo["pct_f2p"].plot(marker="o", ax=ax, color="#2a8a9a", linewidth=2.5, markersize=8)
ax.set_title(f"H4: % anual de lanzamientos F2P (2010-2024) · Spearman rho={spear.statistic:.2f} p={spear.pvalue:.2e}")
ax.set_ylabel("% F2P sobre lanzamientos del anyo"); ax.set_xlabel("Anyo")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(IMG_DIR / "h4_evolucion_f2p.png", dpi=120)
plt.show()

print(por_anyo[["F2P","Pago","total","pct_f2p"]].round(2))


### 4.3 · Multivariante

In [ ]:
num_cols = ["Price","Peak CCU","Positive","Negative","total_reviews","pct_positivas",
            "Average playtime forever","Median playtime forever",
            "Recommendations","Metacritic score","n_plataformas","DLC count"]
num_cols = [c for c in num_cols if c in df.columns]

fig, ax = plt.subplots(figsize=(11, 9))
corr = df[num_cols].corr(method="spearman")
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, ax=ax,
            cbar_kws={"shrink": 0.7})
ax.set_title("Matriz de correlacion (Spearman)")
plt.tight_layout()
plt.savefig(IMG_DIR / "multi_correlacion.png", dpi=120)
plt.show()


---
## Fase 5 · Resultados y conclusiones

### Verificacion de hipotesis

| Hipotesis | Resultado | Evidencia clave |
|-----------|-----------|-----------------|
| H1 - F2P < Pago en %positivas | CONFIRMADA | Mann-Whitney p=1.89e-32 |
| H2 - Distribucion de generos distinta | CONFIRMADA | Chi2=4135 p=0.00e+00 |
| H3 - F2P > Pago en Peak CCU | REFUTADA | Medianas=0; matiz en el top global |
| H4 - %F2P crece desde 2010 | REFUTADA | Curva en U invertida, no monotonia |

### Insights inesperados

- El **boom F2P existio** (2018-2020, coincidente con pandemia y battle royales) pero **se revirtio**: la cuota cayo del 22% al 7% entre 2020 y 2024.
- El modelo F2P es un fenomeno de **cola larga extrema**: 5 de los 10 juegos mas jugados son F2P, pero la mediana de CCU es 0.
- La diferencia de calidad percibida entre modelos es estadisticamente abrumadora pero **modesta en magnitud** (~3 puntos): la narrativa "F2P = juego peor" es una caricatura.

### Recomendaciones

- **Para un estudio indie:** salvo capacidad de sostener live ops durante anyos, el modelo de pago ofrece mejor ratio rentabilidad/riesgo.
- **Para inversores:** el Peak CCU mediano es enganoso. Evaluar exito de un F2P por presencia en el top-100 de CCU mensual.
- **Para diseno de productos:** alinear modelo de negocio con genero. Pelear contra la gravedad del mercado tiene un coste alto.
- **Para futuros analisis:** ampliar con ingresos estimados (owners x precio), retencion (playtime) y nacionalidad del estudio.